# 02 — Save a prediction, then evaluate it

**Completed first-draft walkthrough.** This is a reconstructed 2024 Bahrain example. Qualifying order is a baseline, not a trained model.

In [ ]:
from wearechecking.paths import project_root
from wearechecking.data import latest_snapshot
from wearechecking.features import qualifying_inputs
from wearechecking.student import qualifying_baseline
from wearechecking.learning import check_baseline
from wearechecking.evaluation import save_prediction, evaluate_saved

ROOT = project_root()
snapshot = latest_snapshot(2024, 1, ROOT)
inputs = qualifying_inputs(snapshot, '2024-03-02T12:00:00Z', mode='reconstructed')
display(inputs.frame.head())
print(inputs.metadata['cutoff_utc'])

## Freeze inputs and ranking

Only qualifying identities, team and positions enter this baseline. The feature function never reads race results. It rejects a cutoff before qualifying finishes or at/after the race start.

In [ ]:
prediction = qualifying_baseline(inputs.frame)
print(check_baseline(prediction, inputs.frame))
prediction_folder = save_prediction(prediction, inputs, ROOT)
print('Saved:', prediction_folder.name)
display(prediction.merge(inputs.frame[['driver_id', 'driver']], on='driver_id').head(10))

## Open the labels after saving

Evaluation now reads the race classification and checks that the roster matches. It verifies the saved prediction and input checksums. A substitution cannot silently become another driver's result.

In [ ]:
comparison, metrics = evaluate_saved(prediction_folder, snapshot)
display(comparison.head(10))
metrics

## Interpretation

Position MAE is the average absolute difference between predicted and actual places. Spearman measures order agreement. For this real Bahrain snapshot, MAE is 2.3 and Spearman is about 0.886.

One race is not evidence that a model generalizes. The target includes classified retirements; this is not a prediction of pure pace or a calibrated probability. Read docs/DECISION_NOTE.md, then continue to the chronological benchmark.